In [8]:
# ============================================================================
# SEURAT (.rds/.qs) TO ANNDATA (.h5ad) CONVERSION
# ============================================================================
# This script converts Seurat objects to h5ad format for use with scanpy/Python

library(Seurat)
library(Matrix)
library(data.table)

# Optional: for .qs files
library(qs)

Warning message:
“package ‘Seurat’ was built under R version 4.3.3”
Loading required package: SeuratObject

Loading required package: sp

Warning message:
“package ‘sp’ was built under R version 4.3.3”

Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t


Warning message:
“package ‘Matrix’ was built under R version 4.3.3”
Warning message:
“package ‘qs’ was built under R version 4.3.3”
qs 0.27.3. Announcement: https://github.com/qsbase/qs/issues/103



In [1]:
# ============================================================================
# METHOD 1: MANUAL EXPORT (MOST RELIABLE)
# ============================================================================

# === STEP 1: LOAD SEURAT OBJECT ===
# For .rds files:
#seurat_obj <- readRDS("path/to/your_seurat_object.rds")

# For .qs files (uncomment if using):
# library(qs)
seurat_obj <- readRDS("/fs/ess/PAS2598/Senescence/spatial/GSE233208_Human_visium_ADDS_seurat_processed.rds")

In [2]:
# === STEP 2: SET OUTPUT DIRECTORY ===
output_dir <- "seurat_to_h5ad_export"
dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)

In [3]:
seurat_obj

An object of class Seurat 
36601 features across 115451 samples within 1 assay 
Active assay: Spatial (36601 features, 3500 variable features)
 3 layers present: counts, data, scale.data
 3 dimensional reductions calculated: pca, harmony, umap
 39 images present: slice1, slice1.1, slice1.2, slice1.3, slice1.4, slice1.5, slice1.6, slice1.7, slice1.8, slice1.1.1, slice1.2.1, slice1.3.1, slice1.4.1, slice1.5.1, slice1.6.1, slice1.7.1, slice1.8.1, slice1.9, slice1.10, slice1.11, slice1.12, slice1.13, slice1.14, slice1.15, slice1.16, slice1.1.2, slice1.2.2, slice1.3.2, slice1.4.2, slice1.5.2, slice1.6.2, slice1.7.2, slice1.17, slice1.1.3, slice1.2.3, slice1.3.3, slice1.4.3, slice1.5.3, slice1.6.3

In [4]:
head(seurat_obj@meta.data)

,orig.ident,nCount_Spatial,nFeature_Spatial,SampleID,Case.Year,Case.Num,Diagnosis,Sex,Age,CaptureArea,⋯,bs.q10,annotation,combined_id,ASC_deconv,EX_deconv,INH_deconv,MG_deconv,ODC_deconv,OPC_deconv,VASC_deconv
,<chr>,<dbl>,<int>,<chr>,<int>,<int>,<chr>,<chr>,<int>,<chr>,⋯,<dbl>,<fct>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
AAACAAGTATCTCCCA-1_1_1,SeuratProject,547,291,1,1999,2,Control,F,74,A1,⋯,1,WM3,Oct_2021_1-Control-F-74,0.08382210,0.0000000,0.00000000,0.00000000,0.5871877,0.08380122,0.12419709
AAACAATCTACTAGCA-1_1_1,SeuratProject,242,155,1,1999,2,Control,F,74,A1,⋯,9,L1,Oct_2021_1-Control-F-74,0.16816160,0.1214106,0.00000000,0.00000000,0.0000000,0.30411668,0.33217712
AAACAGAGCGACTCCT-1_1_1,SeuratProject,1602,816,1,1999,2,Control,F,74,A1,⋯,1,WM3,Oct_2021_1-Control-F-74,0.24050308,0.0000000,0.08961442,0.10921104,0.3882161,0.00000000,0.15765875
AAACATTTCCCGGATT-1_1_1,SeuratProject,1025,599,1,1999,2,Control,F,74,A1,⋯,2,L5/6,Oct_2021_1-Control-F-74,0.20749189,0.2359938,0.14245138,0.00000000,0.1111000,0.13684185,0.11417334
AAACCACTACACAGAT-1_1_1,SeuratProject,283,158,1,1999,2,Control,F,74,A1,⋯,9,L1,Oct_2021_1-Control-F-74,0.13427857,0.1711590,0.00000000,0.09074438,0.3931475,0.09545940,0.09950331
AAACCCGAACGAAATC-1_1_1,SeuratProject,1229,633,1,1999,2,Control,F,74,A1,⋯,1,WM3,Oct_2021_1-Control-F-74,0.09670091,0.0000000,0.00000000,0.17976665,0.3598749,0.11332994,0.16276062


In [ ]:
# non-spatial

# === STEP 3: EXTRACT AND EXPORT COUNT MATRIX ===
# Get the counts from the RNA assay
DefaultAssay(seurat_obj) <- 'RNA'

# Export raw counts (genes x cells)
counts <- GetAssayData(seurat_obj, slot = "counts")

# Export to Matrix Market format
writeMM(counts, file.path(output_dir, "matrix.mtx"))

# Compress the matrix file
system(paste0("gzip -f ", file.path(output_dir, "matrix.mtx")))

cat("✓ Count matrix exported\n")

# === STEP 4: EXPORT FEATURES (GENES) ===
features <- data.frame(
  gene_id = rownames(seurat_obj),
  gene_name = rownames(seurat_obj),
  feature_type = "Gene Expression"
)

fwrite(features, 
       file.path(output_dir, "features.tsv.gz"),
       sep = "\t", 
       col.names = FALSE,
       compress = "gzip")

cat("✓ Features exported\n")

# === STEP 5: EXPORT BARCODES (CELLS) ===
barcodes <- data.frame(barcode = colnames(seurat_obj))

fwrite(barcodes,
       file.path(output_dir, "barcodes.tsv.gz"),
       sep = "\t",
       col.names = FALSE,
       compress = "gzip")

cat("✓ Barcodes exported\n")

# === STEP 6: EXPORT METADATA (OBSERVATIONS) ===
metadata <- seurat_obj@meta.data
metadata$barcode <- rownames(metadata)

fwrite(metadata,
       file.path(output_dir, "metadata.csv"),
       row.names = FALSE)

cat("✓ Metadata exported\n")

# === STEP 7: EXPORT VARIABLE FEATURES (OPTIONAL) ===
if(length(VariableFeatures(seurat_obj)) > 0) {
  var_features <- data.frame(gene = VariableFeatures(seurat_obj))
  fwrite(var_features,
         file.path(output_dir, "variable_features.csv"),
         row.names = FALSE)
  cat("✓ Variable features exported\n")
}

# === STEP 8: EXPORT REDUCTIONS (PCA, UMAP, etc.) ===
reductions <- Reductions(seurat_obj)
if(length(reductions) > 0) {
  for(reduction in reductions) {
    reduction_data <- Embeddings(seurat_obj, reduction = reduction)
    fwrite(as.data.frame(reduction_data),
           file.path(output_dir, paste0(reduction, "_embeddings.csv")),
           row.names = TRUE)
  }
  cat(paste0("✓ Exported ", length(reductions), " reductions\n"))
}

# === STEP 9: EXPORT ADDITIONAL LAYERS (OPTIONAL) ===
# Export normalized data if available
#if("data" %in% slotNames(seurat_obj@assays$RNA)) {
#  normalized_data <- GetAssayData(seurat_obj, slot = "data")
#  writeMM(normalized_data, file.path(output_dir, "normalized_matrix.mtx"))
#  system(paste0("gzip -f ", file.path(output_dir, "normalized_matrix.mtx")))
#  cat("✓ Normalized data exported\n")
#}

# Export scaled data if available
#if("scale.data" %in% slotNames(seurat_obj@assays$RNA) && 
#   nrow(GetAssayData(seurat_obj, slot = "scale.data")) > 0) {
#  scaled_data <- GetAssayData(seurat_obj, slot = "scale.data")
#  writeMM(scaled_data, file.path(output_dir, "scaled_matrix.mtx"))
#  system(paste0("gzip -f ", file.path(output_dir, "scaled_matrix.mtx")))
#  cat("✓ Scaled data exported\n")
#}

cat("\n=== R EXPORT COMPLETE ===\n")
cat(paste0("Files saved to: ", output_dir, "\n"))
cat("\nNext: Run the Python script to create .h5ad file\n")

In [9]:
# spatial

# === STEP 3: EXTRACT AND EXPORT COUNT MATRIX ===
# Get the counts from the Spatial assay
DefaultAssay(seurat_obj) <- 'Spatial'

# Export raw counts (genes x cells)
counts <- GetAssayData(seurat_obj, slot = "counts")

# Export to Matrix Market format
writeMM(counts, file.path(output_dir, "matrix.mtx"))

# Compress the matrix file
system(paste0("gzip -f ", file.path(output_dir, "matrix.mtx")))
cat("✓ Count matrix exported\n")

# === STEP 4: EXPORT FEATURES (GENES) ===
features <- data.frame(
  gene_id = rownames(seurat_obj),
  gene_name = rownames(seurat_obj),
  feature_type = "Gene Expression"
)
fwrite(features, 
       file.path(output_dir, "features.tsv.gz"),
       sep = "\t", 
       col.names = FALSE,
       compress = "gzip")
cat("✓ Features exported\n")

# === STEP 5: EXPORT BARCODES (CELLS) ===
barcodes <- data.frame(barcode = colnames(seurat_obj))
fwrite(barcodes,
       file.path(output_dir, "barcodes.tsv.gz"),
       sep = "\t",
       col.names = FALSE,
       compress = "gzip")
cat("✓ Barcodes exported\n")

# === STEP 6: EXPORT METADATA (OBSERVATIONS) ===
metadata <- seurat_obj@meta.data
metadata$barcode <- rownames(metadata)
fwrite(metadata,
       file.path(output_dir, "metadata.csv"),
       row.names = FALSE)
cat("✓ Metadata exported\n")

# === STEP 7: EXPORT VARIABLE FEATURES (OPTIONAL) ===
if(length(VariableFeatures(seurat_obj)) > 0) {
  var_features <- data.frame(gene = VariableFeatures(seurat_obj))
  fwrite(var_features,
         file.path(output_dir, "variable_features.csv"),
         row.names = FALSE)
  cat("✓ Variable features exported\n")
}

# === STEP 8: EXPORT REDUCTIONS (PCA, UMAP, etc.) ===
reductions <- Reductions(seurat_obj)
if(length(reductions) > 0) {
  for(reduction in reductions) {
    reduction_data <- Embeddings(seurat_obj, reduction = reduction)
    fwrite(as.data.frame(reduction_data),
           file.path(output_dir, paste0(reduction, "_embeddings.csv")),
           row.names = TRUE)
  }
  cat(paste0("✓ Exported ", length(reductions), " reductions\n"))
}

cat("\n=== R EXPORT COMPLETE ===\n")
cat(paste0("Files saved to: ", output_dir, "\n"))
cat("\nNext: Run the Python script to create .h5ad file\n")

Warning message:
“The `slot` argument of `GetAssayData()` is deprecated as of SeuratObject 5.0.0.
ℹ Please use the `layer` argument instead.”


NULL

✓ Count matrix exported
✓ Features exported
✓ Barcodes exported
✓ Metadata exported
✓ Variable features exported
✓ Exported 3 reductions

=== R EXPORT COMPLETE ===
Files saved to: seurat_to_h5ad_export

Next: Run the Python script to create .h5ad file


In [ ]:
#!/usr/bin/env python3
"""
Simple script to create h5ad from Seurat export
"""

import scanpy as sc
import pandas as pd
import scipy.io
import os

# === CONFIGURATION ===
input_dir = "/users/PAS2598/ggaitos/2025/scripts/single-cell/deepsas/psychad_samples/seurat_to_h5ad_export"
output_file = "seurat_converted.h5ad"

print("Loading files...")

# === LOAD MATRIX ===
matrix = scipy.io.mmread(os.path.join(input_dir, "matrix.mtx.gz"))
matrix = matrix.T.tocsr()  # Transpose to cells x genes

print(f"Matrix: {matrix.shape[0]} cells x {matrix.shape[1]} genes")

# === LOAD BARCODES ===
barcodes = pd.read_csv(
    os.path.join(input_dir, "barcodes.tsv.gz"),
    sep="\t",
    header=None,
    names=["barcode"]
)

# === LOAD FEATURES ===
features = pd.read_csv(
    os.path.join(input_dir, "features.tsv.gz"),
    sep="\t",
    header=None,
    names=["gene_id", "gene_name", "feature_type"]
)

# === CREATE ANNDATA ===
adata = sc.AnnData(
    X=matrix,
    obs=barcodes.set_index("barcode"),
    var=features.set_index("gene_name")
)

# === LOAD METADATA (OPTIONAL) ===
metadata_file = os.path.join(input_dir, "metadata.csv")
if os.path.exists(metadata_file):
    metadata = pd.read_csv(metadata_file).set_index("barcode")
    adata.obs = adata.obs.join(metadata, how="left")
    print(f"Metadata: {len(adata.obs.columns)} columns")

# === SAVE ===
adata.write_h5ad(output_file)

print(f"\n✓ Success! Created: {output_file}")
print(f"  {adata.n_obs:,} cells")
print(f"  {adata.n_vars:,} genes")

In [1]:
#!/usr/bin/env python3
"""
Simple script to create h5ad from Seurat export
"""
import scanpy as sc
import pandas as pd
import scipy.io
import os
import numpy as np

# === CONFIGURATION ===
input_dir = "/users/PAS2598/ggaitos/2025/scripts/single-cell/deepsas/psychad_samples/seurat_to_h5ad_export"
output_file = "gse233208_spatial.h5ad"

print("Loading files...")

# === LOAD MATRIX ===
matrix = scipy.io.mmread(os.path.join(input_dir, "matrix.mtx.gz"))
matrix = matrix.T.tocsr()  # Transpose to cells x genes
print(f"Matrix: {matrix.shape[0]} cells x {matrix.shape[1]} genes")

# === LOAD BARCODES ===
barcodes = pd.read_csv(
    os.path.join(input_dir, "barcodes.tsv.gz"),
    sep="\t",
    header=None,
    names=["barcode"]
)

# === LOAD FEATURES ===
features = pd.read_csv(
    os.path.join(input_dir, "features.tsv.gz"),
    sep="\t",
    header=None,
    names=["gene_id", "gene_name", "feature_type"]
)

# === CREATE ANNDATA ===
adata = sc.AnnData(
    X=matrix,
    obs=barcodes.set_index("barcode"),
    var=features.set_index("gene_name")
)

# === LOAD METADATA (OPTIONAL) ===
metadata_file = os.path.join(input_dir, "metadata.csv")
if os.path.exists(metadata_file):
    # Load with low_memory=False to avoid dtype warning
    metadata = pd.read_csv(metadata_file, low_memory=False).set_index("barcode")
    
    print(f"Metadata: {len(metadata.columns)} columns")
    print("Cleaning metadata for h5ad compatibility...")
    
    # Clean each column
    for col in metadata.columns:
        # Check if column has mixed types or problematic data
        if metadata[col].dtype == 'object':
            try:
                # Try to convert to numeric if possible
                converted = pd.to_numeric(metadata[col], errors='coerce')
                if converted.notna().sum() > 0:  # If some values converted successfully
                    metadata[col] = converted
                else:
                    # Keep as string, but handle None/NaN
                    metadata[col] = metadata[col].fillna('').astype(str)
            except:
                # Convert to string as fallback
                metadata[col] = metadata[col].fillna('').astype(str)
        
        # Handle any remaining None values
        if metadata[col].isna().any():
            if metadata[col].dtype in ['float64', 'float32', 'int64', 'int32']:
                metadata[col] = metadata[col].fillna(0)
            else:
                metadata[col] = metadata[col].fillna('')
    
    # Join metadata
    adata.obs = adata.obs.join(metadata, how="left")
    print(f"Added metadata: {len(adata.obs.columns)} columns total")

# === SAVE ===
print("Saving h5ad file...")
adata.write_h5ad(output_file)

print(f"\n✓ Success! Created: {output_file}")
print(f"  {adata.n_obs:,} cells")
print(f"  {adata.n_vars:,} genes")

Loading files...
Matrix: 115451 cells x 36601 genes
Metadata: 40 columns
Cleaning metadata for h5ad compatibility...
Added metadata: 40 columns total
Saving h5ad file...

✓ Success! Created: gse233208_spatial.h5ad
  115,451 cells
  36,601 genes
